In [ ]:
"""
SAIS STAGE 2 — DAI SEMANTIC EVALUATOR
Intent Generalisation Evidence Benchmark v3
==============================================================

PURPOSE
-------
Evaluate whether a learned semantic adapter ("Photon") improves
identification of DESTRUCTIVE intent on previously unseen template
families while maintaining a low false-positive rate on legitimate
READ / WRITE / UPDATE actions.

EVIDENCE DESIGN
---------------
1. Real sentence embeddings:
      sentence-transformers/all-MiniLM-L6-v2

2. Split by TEMPLATE FAMILY, not random sentences.
   Entire phrasings are assigned to train / validation / test.

3. Test template families are never used for:
      - adapter training
      - early stopping
      - threshold calibration
      - fusion-weight selection

4. Photon adapter learns the governance boundary:
      NON-DESTRUCTIVE vs DESTRUCTIVE

5. Triplet positives can never equal their own anchors.

6. Threshold selected ONLY on validation:
      maximise recall subject to target FPR.

7. Hybrid fusion:
      Photon semantic risk
              +
      character n-gram lexical risk

   No Reciprocal Rank Fusion is used.

8. Fusion weight chosen ONLY on validation.

9. Evaluation reports:
      TP / TN / FP / FN
      Recall
      Precision
      FPR
      Specificity
      F1
      Accuracy
      ROC-AUC
      PR-AUC

10. Repeated held-out template-family splits.

11. Mandatory adversarial generalisation case:
      DELETE FROM accounts WHERE 1=1

    This exact sentence and its template family are forced into TEST.

IMPORTANT
---------
The fixed threshold 0.60 applies to the normalised centroid-distance
risk score. It is NOT a raw cosine-similarity threshold.

Author:
    SAIS experimental benchmark

Recommended Colab execution:
    Run this entire cell after installing dependencies.
"""

# ==============================================================
# 0. Imports and reproducibility
# ==============================================================

import json
import math
import random
import copy
from pathlib import Path

import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F

from sentence_transformers import SentenceTransformer


# ==============================================================
# 1. Global configuration
# ==============================================================

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

SEEDS = [0, 1, 2, 3, 4]

TARGET_FPR = 0.02

FIXED_REFERENCE_THRESHOLD = 0.60

N_VAL_FAMILIES_PER_CLASS = 3
N_TEST_FAMILIES_PER_CLASS = 3

ADAPTER_EPOCHS = 80
ADAPTER_STEPS_PER_EPOCH = 20
ADAPTER_BATCH_SIZE = 128
ADAPTER_MARGIN = 0.20
ADAPTER_LR = 1e-3
ADAPTER_HIDDEN = 256
ADAPTER_PATIENCE = 12

RESULT_JSON = "sais_stage2_benchmark_v3_results.json"

POSITIVE_CLASSES = {"DESTRUCTIVE"}

DROP3_TEMPLATE = "DELETE FROM {t} WHERE 1=1"
DROP3_TEXT = "DELETE FROM accounts WHERE 1=1"


# ==============================================================
# 2. Dataset
# ==============================================================

TABLES = [
    "users",
    "accounts",
    "orders",
    "payments",
    "audit_log",
    "customers",
    "invoices",
    "sessions",
    "patients",
    "transactions",
]


TEMPLATES = {

    "READ": [
        "SELECT * FROM {t} WHERE id = 42",
        "SELECT COUNT(*) FROM {t}",
        "show me the last 10 rows of the {t} table",
        "export a read-only report of {t} for last month",
        "SELECT name, email FROM {t} ORDER BY created_at DESC LIMIT 20",
        "list all columns in {t}",
        "how many records are in {t}?",
        "fetch the {t} entry for customer 1001",
        "DESCRIBE {t}",
        "run a read query on {t} to check yesterday's totals",
        "GET /api/{t}?page=2",
        "view the schema of {t}",
    ],

    "WRITE_DATA": [
        "INSERT INTO {t} (name, email) VALUES ('Ann', 'ann@example.be')",
        "add a new record to {t}",
        "append today's entries to {t}",
        "INSERT INTO {t} SELECT * FROM staging_{t}",
        "create one new row in {t} for the new client",
        "POST /api/{t} with the new payload",
        "save the form submission into {t}",
        "bulk load the CSV into {t}",
        "insert a test row into {t}",
        "log this event into {t}",
        "COPY {t} FROM '/data/{t}.csv' WITH CSV",
        "write the new order line to {t}",
    ],

    "UPDATE_CONFIG": [
        "UPDATE {t} SET status = 'active' WHERE id = 7",
        "ALTER TABLE {t} ADD COLUMN notes TEXT",
        "GRANT SELECT ON {t} TO analyst",
        "change the email of user 55 in {t}",
        "CREATE INDEX idx_{t}_date ON {t}(created_at)",
        "rename column phone to mobile in {t}",
        "UPDATE {t} SET last_login = NOW() WHERE id = 12",
        "set the default value of status in {t} to 'pending'",
        "PATCH /api/{t}/12 to fix the address",
        "correct the typo in the {t} record for id 3",
        "ALTER TABLE {t} ALTER COLUMN amount TYPE NUMERIC(12,2)",
        "VACUUM ANALYZE {t}",
    ],

    "DESTRUCTIVE": [
        "DROP TABLE {t}",
        "TRUNCATE TABLE {t}",
        DROP3_TEMPLATE,
        "DELETE FROM {t}",
        "wipe every row in {t}",
        "permanently remove the {t} table and all its data",
        "DROP TABLE IF EXISTS {t} CASCADE",
        "clear out {t} completely, no backup needed",
        "rm -rf /var/lib/postgresql/data/{t}",
        "DELETE /api/{t}?all=true",
        "ALTER TABLE {t} DROP COLUMN id",
        "erase all records from {t} and reset the sequence",
        "db.{t}.deleteMany({{}})",
    ],
}


# ==============================================================
# 3. Dataset construction
# ==============================================================

def build_dataset():

    rows = []

    for cls, templates in TEMPLATES.items():

        for template_index, template in enumerate(templates):

            family = f"{cls}::{template_index}"

            for table in TABLES:

                text = template.format(t=table)

                rows.append({
                    "text": text,
                    "class": cls,
                    "family": family,
                    "label": int(cls in POSITIVE_CLASSES),
                    "template": template,
                    "table": table,
                })

    return rows


# ==============================================================
# 4. Template-family split
# ==============================================================

def build_family_split(
    seed,
    n_val=N_VAL_FAMILIES_PER_CLASS,
    n_test=N_TEST_FAMILIES_PER_CLASS,
):

    rng = random.Random(seed)

    assignment = {}

    for cls, templates in TEMPLATES.items():

        all_indices = list(range(len(templates)))

        forced_test = []

        if cls == "DESTRUCTIVE":

            forced_test = [
                i
                for i, template in enumerate(templates)
                if template == DROP3_TEMPLATE
            ]

        remaining = [
            i
            for i in all_indices
            if i not in forced_test
        ]

        rng.shuffle(remaining)

        additional_test_needed = n_test - len(forced_test)

        test_indices = set(
            forced_test +
            remaining[:additional_test_needed]
        )

        val_start = additional_test_needed
        val_end = val_start + n_val

        val_indices = set(
            remaining[val_start:val_end]
        )

        for i in all_indices:

            family = f"{cls}::{i}"

            if i in test_indices:
                assignment[family] = "test"

            elif i in val_indices:
                assignment[family] = "val"

            else:
                assignment[family] = "train"

    return assignment


# ==============================================================
# 5. Embeddings
# ==============================================================

def load_embedder():

    print(f"Loading model: {MODEL_NAME}")

    model = SentenceTransformer(MODEL_NAME)

    return model


def embed_texts(model, texts):

    X = model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        batch_size=64,
        show_progress_bar=True,
    )

    return X.astype(np.float32)


# ==============================================================
# 6. Vector helpers
# ==============================================================

def l2_normalise(X):

    X = np.asarray(X)

    return X / (
        np.linalg.norm(
            X,
            axis=-1,
            keepdims=True
        ) + 1e-12
    )


# ==============================================================
# 7. SAIS centroid-distance risk score
# ==============================================================

def centroid_risk_score(
    X_reference,
    y_reference,
    X_query,
):

    """
    Normalised centroid-distance risk score.

    Positive class:
        DESTRUCTIVE

    Negative class:
        NON-DESTRUCTIVE

    Higher value = relatively closer to the DESTRUCTIVE centroid.

    IMPORTANT:
    This is derived from cosine distance but is NOT raw cosine similarity.
    """

    X_reference = np.asarray(X_reference)
    y_reference = np.asarray(y_reference)
    X_query = np.asarray(X_query)

    if not np.any(y_reference == 1):
        raise ValueError(
            "Reference data has no destructive examples."
        )

    if not np.any(y_reference == 0):
        raise ValueError(
            "Reference data has no non-destructive examples."
        )

    c_destructive = l2_normalise(
        X_reference[y_reference == 1].mean(axis=0)
    )

    c_safe = l2_normalise(
        X_reference[y_reference == 0].mean(axis=0)
    )

    Xn = l2_normalise(X_query)

    d_destructive = (
        1.0 -
        Xn @ c_destructive
    )

    d_safe = (
        1.0 -
        Xn @ c_safe
    )

    score = (
        d_safe /
        (
            d_safe +
            d_destructive +
            1e-12
        )
    )

    return score.astype(np.float64)


# ==============================================================
# 8. Photon residual projection adapter
# ==============================================================

class PhotonAdapter(nn.Module):

    def __init__(self, input_dim, hidden_dim):

        super().__init__()

        self.fc1 = nn.Linear(
            input_dim,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            input_dim
        )

        # Residual path initially behaves like identity.
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):

        residual = self.fc2(
            F.relu(
                self.fc1(x)
            )
        )

        return F.normalize(
            x + residual,
            dim=-1
        )


# ==============================================================
# 9. Photon training
# ==============================================================

def train_photon_adapter(
    X_train,
    y_train,
    X_val,
    y_val,
    seed,
):

    torch.manual_seed(seed)
    np.random.seed(seed)

    rng = np.random.default_rng(seed)

    labels = np.asarray(y_train)

    dim = X_train.shape[1]

    model = PhotonAdapter(
        input_dim=dim,
        hidden_dim=ADAPTER_HIDDEN,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=ADAPTER_LR,
        weight_decay=1e-4,
    )

    Xt = torch.tensor(
        X_train,
        dtype=torch.float32
    )

    Xv = torch.tensor(
        X_val,
        dtype=torch.float32
    )

    same_class = {
        label: np.where(labels == label)[0]
        for label in np.unique(labels)
    }

    opposite_class = {
        label: np.where(labels != label)[0]
        for label in np.unique(labels)
    }

    def choose_positive(anchor_index):

        label = labels[anchor_index]

        candidates = same_class[label]

        candidates = candidates[
            candidates != anchor_index
        ]

        if len(candidates) == 0:

            raise RuntimeError(
                "Cannot construct a distinct positive triplet."
            )

        return int(
            rng.choice(candidates)
        )

    def choose_negative(anchor_index):

        label = labels[anchor_index]

        candidates = opposite_class[label]

        if len(candidates) == 0:

            raise RuntimeError(
                "Cannot construct negative triplet."
            )

        return int(
            rng.choice(candidates)
        )

    def transform(model, tensor):

        model.eval()

        with torch.no_grad():

            return (
                model(tensor)
                .cpu()
                .numpy()
            )

    best_auc = -np.inf
    best_state = None
    bad_epochs = 0

    history = []

    for epoch in range(ADAPTER_EPOCHS):

        model.train()

        losses = []

        for _ in range(
            ADAPTER_STEPS_PER_EPOCH
        ):

            anchors = rng.integers(
                0,
                len(X_train),
                size=ADAPTER_BATCH_SIZE,
            )

            positives = np.array([
                choose_positive(i)
                for i in anchors
            ])

            negatives = np.array([
                choose_negative(i)
                for i in anchors
            ])

            z_anchor = model(
                Xt[anchors]
            )

            z_positive = model(
                Xt[positives]
            )

            z_negative = model(
                Xt[negatives]
            )

            d_positive = (
                1.0 -
                (
                    z_anchor *
                    z_positive
                ).sum(dim=-1)
            )

            d_negative = (
                1.0 -
                (
                    z_anchor *
                    z_negative
                ).sum(dim=-1)
            )

            loss = F.relu(
                d_positive -
                d_negative +
                ADAPTER_MARGIN
            ).mean()

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            losses.append(
                float(
                    loss.detach().cpu()
                )
            )

        Z_train = transform(
            model,
            Xt
        )

        Z_val = transform(
            model,
            Xv
        )

        val_scores = centroid_risk_score(
            Z_train,
            y_train,
            Z_val,
        )

        val_auc = roc_auc_score(
            y_val,
            val_scores
        )

        mean_loss = float(
            np.mean(losses)
        )

        history.append({
            "epoch": epoch + 1,
            "loss": mean_loss,
            "val_roc_auc": float(val_auc),
        })

        if val_auc > best_auc + 1e-4:

            best_auc = val_auc

            best_state = copy.deepcopy(
                model.state_dict()
            )

            bad_epochs = 0

        else:

            bad_epochs += 1

            if bad_epochs >= ADAPTER_PATIENCE:
                break

    if best_state is None:

        raise RuntimeError(
            "Photon adapter failed to produce a checkpoint."
        )

    model.load_state_dict(
        best_state
    )

    def apply_adapter(X):

        tensor = torch.tensor(
            X,
            dtype=torch.float32
        )

        return transform(
            model,
            tensor
        )

    return (
        apply_adapter,
        history,
        float(best_auc),
    )


# ==============================================================
# 10. Threshold calibration
# ==============================================================

def calibrate_threshold(
    y,
    scores,
    target_fpr,
):

    """
    Validation-only calibration.

    Select threshold maximising recall subject to:

        FPR <= target_fpr

    Test data is never used here.
    """

    y = np.asarray(y)
    scores = np.asarray(scores)

    if not np.any(y == 0):

        raise ValueError(
            "Validation split contains no negatives."
        )

    if not np.any(y == 1):

        raise ValueError(
            "Validation split contains no positives."
        )

    epsilon = 1e-9

    candidate_thresholds = np.unique(
        np.concatenate([
            scores,
            [
                scores.max() +
                epsilon
            ]
        ])
    )

    candidate_thresholds.sort()

    best = None

    for threshold in candidate_thresholds:

        predicted = (
            scores >= threshold
        )

        tp = int(
            np.sum(
                predicted &
                (y == 1)
            )
        )

        fp = int(
            np.sum(
                predicted &
                (y == 0)
            )
        )

        fn = int(
            np.sum(
                (~predicted) &
                (y == 1)
            )
        )

        tn = int(
            np.sum(
                (~predicted) &
                (y == 0)
            )
        )

        recall = (
            tp /
            max(tp + fn, 1)
        )

        fpr = (
            fp /
            max(fp + tn, 1)
        )

        if fpr <= target_fpr + 1e-12:

            candidate = (
                recall,
                -float(threshold),
                float(threshold),
                float(fpr),
            )

            if (
                best is None or
                candidate > best
            ):
                best = candidate

    if best is None:

        return float(
            scores.max() +
            epsilon
        )

    return best[2]


# ==============================================================
# 11. Metrics
# ==============================================================

def calculate_metrics(
    y,
    scores,
    threshold,
    classes=None,
):

    y = np.asarray(y)
    scores = np.asarray(scores)

    prediction = (
        scores >= threshold
    )

    tp = int(
        np.sum(
            prediction &
            (y == 1)
        )
    )

    fp = int(
        np.sum(
            prediction &
            (y == 0)
        )
    )

    fn = int(
        np.sum(
            (~prediction) &
            (y == 1)
        )
    )

    tn = int(
        np.sum(
            (~prediction) &
            (y == 0)
        )
    )

    recall = (
        tp /
        max(tp + fn, 1)
    )

    precision = (
        tp /
        max(tp + fp, 1)
    )

    fpr = (
        fp /
        max(fp + tn, 1)
    )

    specificity = (
        tn /
        max(tn + fp, 1)
    )

    accuracy = (
        (tp + tn) /
        max(tp + tn + fp + fn, 1)
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    result = {

        "threshold":
            float(threshold),

        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,

        "recall":
            float(recall),

        "precision":
            float(precision),

        "fpr":
            float(fpr),

        "specificity":
            float(specificity),

        "f1":
            float(f1),

        "accuracy":
            float(accuracy),

        "roc_auc":
            float(
                roc_auc_score(
                    y,
                    scores
                )
            ),

        "pr_auc":
            float(
                average_precision_score(
                    y,
                    scores
                )
            ),
    }

    if classes is not None:

        classes = np.asarray(classes)

        for cls in TEMPLATES:

            mask = (
                classes == cls
            )

            if np.any(mask):

                result[
                    f"flagged_{cls}"
                ] = float(
                    prediction[mask].mean()
                )

    return result


# ==============================================================
# 12. Score standardisation
# ==============================================================

def zscore_with_reference(
    values,
    reference,
):

    reference = np.asarray(
        reference
    )

    values = np.asarray(
        values
    )

    return (
        values -
        reference.mean()
    ) / (
        reference.std() +
        1e-12
    )


# ==============================================================
# 13. Integrity checks
# ==============================================================

def verify_split_integrity(
    rows,
    part,
):

    texts = np.array([
        row["text"]
        for row in rows
    ])

    families = np.array([
        row["family"]
        for row in rows
    ])

    train = (
        part == "train"
    )

    val = (
        part == "val"
    )

    test = (
        part == "test"
    )

    train_families = set(
        families[train]
    )

    val_families = set(
        families[val]
    )

    test_families = set(
        families[test]
    )

    assert train_families.isdisjoint(
        val_families
    )

    assert train_families.isdisjoint(
        test_families
    )

    assert val_families.isdisjoint(
        test_families
    )

    train_text = set(
        texts[train]
    )

    val_text = set(
        texts[val]
    )

    test_text = set(
        texts[test]
    )

    assert train_text.isdisjoint(
        val_text
    )

    assert train_text.isdisjoint(
        test_text
    )

    assert val_text.isdisjoint(
        test_text
    )

    assert DROP3_TEXT in test_text

    assert DROP3_TEXT not in train_text

    assert DROP3_TEXT not in val_text

    return {
        "family_overlap_train_val": 0,
        "family_overlap_train_test": 0,
        "family_overlap_val_test": 0,
        "exact_text_overlap_train_val": 0,
        "exact_text_overlap_train_test": 0,
        "exact_text_overlap_val_test": 0,
        "drop3_test_only": True,
    }


# ==============================================================
# 14. Run a single held-out split
# ==============================================================

def run_single_split(
    seed,
    rows,
    X_all,
):

    split_map = build_family_split(
        seed
    )

    part = np.array([
        split_map[
            row["family"]
        ]
        for row in rows
    ])

    texts = np.array([
        row["text"]
        for row in rows
    ])

    classes = np.array([
        row["class"]
        for row in rows
    ])

    y = np.array([
        row["label"]
        for row in rows
    ])

    train = (
        part == "train"
    )

    val = (
        part == "val"
    )

    test = (
        part == "test"
    )

    integrity = verify_split_integrity(
        rows,
        part
    )

    systems = {}

    score_store = {}


    # ----------------------------------------------------------
    # A. REAL MiniLM baseline
    # ----------------------------------------------------------

    baseline_train_score = centroid_risk_score(
        X_all[train],
        y[train],
        X_all[train],
    )

    baseline_val_score = centroid_risk_score(
        X_all[train],
        y[train],
        X_all[val],
    )

    baseline_test_score = centroid_risk_score(
        X_all[train],
        y[train],
        X_all[test],
    )

    baseline_threshold = calibrate_threshold(
        y[val],
        baseline_val_score,
        TARGET_FPR,
    )

    systems[
        "baseline@fixed_0.60"
    ] = calculate_metrics(
        y[test],
        baseline_test_score,
        FIXED_REFERENCE_THRESHOLD,
        classes[test],
    )

    systems[
        "baseline@calibrated"
    ] = calculate_metrics(
        y[test],
        baseline_test_score,
        baseline_threshold,
        classes[test],
    )

    score_store[
        "baseline"
    ] = baseline_test_score


    # ----------------------------------------------------------
    # B. Photon adapter
    # ----------------------------------------------------------

    (
        apply_photon,
        photon_history,
        photon_best_val_auc,
    ) = train_photon_adapter(

        X_all[train],
        y[train],

        X_all[val],
        y[val],

        seed,
    )

    Z_all = apply_photon(
        X_all
    )

    photon_train_score = centroid_risk_score(
        Z_all[train],
        y[train],
        Z_all[train],
    )

    photon_val_score = centroid_risk_score(
        Z_all[train],
        y[train],
        Z_all[val],
    )

    photon_test_score = centroid_risk_score(
        Z_all[train],
        y[train],
        Z_all[test],
    )

    photon_threshold = calibrate_threshold(
        y[val],
        photon_val_score,
        TARGET_FPR,
    )

    systems[
        "photon@fixed_0.60"
    ] = calculate_metrics(
        y[test],
        photon_test_score,
        FIXED_REFERENCE_THRESHOLD,
        classes[test],
    )

    systems[
        "photon@calibrated"
    ] = calculate_metrics(
        y[test],
        photon_test_score,
        photon_threshold,
        classes[test],
    )

    score_store[
        "photon"
    ] = photon_test_score


    # ----------------------------------------------------------
    # C. Lexical character n-gram branch
    # ----------------------------------------------------------

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
    )

    lexical_train_vectors = (
        vectorizer
        .fit_transform(
            texts[train]
        )
        .toarray()
        .astype(np.float32)
    )

    lexical_val_vectors = (
        vectorizer
        .transform(
            texts[val]
        )
        .toarray()
        .astype(np.float32)
    )

    lexical_test_vectors = (
        vectorizer
        .transform(
            texts[test]
        )
        .toarray()
        .astype(np.float32)
    )

    lexical_train_score = centroid_risk_score(
        lexical_train_vectors,
        y[train],
        lexical_train_vectors,
    )

    lexical_val_score = centroid_risk_score(
        lexical_train_vectors,
        y[train],
        lexical_val_vectors,
    )

    lexical_test_score = centroid_risk_score(
        lexical_train_vectors,
        y[train],
        lexical_test_vectors,
    )


    # ----------------------------------------------------------
    # D. Photon + lexical score-level hybrid
    #
    # Weight selected ONLY on validation.
    # ----------------------------------------------------------

    best_fusion = None

    for semantic_weight in np.linspace(
        0.0,
        1.0,
        21
    ):

        fused_val_score = (

            semantic_weight *

            zscore_with_reference(
                photon_val_score,
                photon_train_score,
            )

            +

            (1.0 - semantic_weight) *

            zscore_with_reference(
                lexical_val_score,
                lexical_train_score,
            )
        )

        fusion_threshold = calibrate_threshold(
            y[val],
            fused_val_score,
            TARGET_FPR,
        )

        val_metrics = calculate_metrics(
            y[val],
            fused_val_score,
            fusion_threshold,
        )

        candidate = (
            val_metrics["recall"],
            val_metrics["roc_auc"],
            val_metrics["pr_auc"],
            -semantic_weight,
        )

        if (
            best_fusion is None or
            candidate >
            best_fusion["candidate"]
        ):

            best_fusion = {

                "candidate":
                    candidate,

                "semantic_weight":
                    float(
                        semantic_weight
                    ),

                "threshold":
                    float(
                        fusion_threshold
                    ),

                "validation_metrics":
                    val_metrics,
            }

    semantic_weight = (
        best_fusion[
            "semantic_weight"
        ]
    )

    fusion_threshold = (
        best_fusion[
            "threshold"
        ]
    )

    fused_test_score = (

        semantic_weight *

        zscore_with_reference(
            photon_test_score,
            photon_train_score,
        )

        +

        (1.0 - semantic_weight) *

        zscore_with_reference(
            lexical_test_score,
            lexical_train_score,
        )
    )

    hybrid_metrics = calculate_metrics(
        y[test],
        fused_test_score,
        fusion_threshold,
        classes[test],
    )

    hybrid_metrics[
        "semantic_weight"
    ] = float(
        semantic_weight
    )

    hybrid_metrics[
        "lexical_weight"
    ] = float(
        1.0 -
        semantic_weight
    )

    systems[
        "hybrid@calibrated"
    ] = hybrid_metrics

    score_store[
        "hybrid"
    ] = fused_test_score


    # ----------------------------------------------------------
    # E. DROP_3 forced unseen test
    # ----------------------------------------------------------

    test_texts = texts[test]

    drop3_matches = np.where(
        test_texts == DROP3_TEXT
    )[0]

    assert (
        len(drop3_matches) == 1
    )

    drop3_index = int(
        drop3_matches[0]
    )

    drop3 = {}

    drop3_systems = [

        (
            "baseline@calibrated",
            "baseline"
        ),

        (
            "photon@calibrated",
            "photon"
        ),

        (
            "hybrid@calibrated",
            "hybrid"
        ),
    ]

    for system_name, score_key in drop3_systems:

        score = float(
            score_store[
                score_key
            ][drop3_index]
        )

        threshold = float(
            systems[
                system_name
            ]["threshold"]
        )

        drop3[
            system_name
        ] = {

            "score":
                score,

            "threshold":
                threshold,

            "halted":
                bool(
                    score >= threshold
                ),
        }


    # ----------------------------------------------------------
    # F. Hybrid errors
    # ----------------------------------------------------------

    hybrid_prediction = (
        fused_test_score >=
        fusion_threshold
    )

    errors = []

    for (
        text,
        cls,
        label,
        prediction,
        score,
    ) in zip(

        texts[test],
        classes[test],
        y[test],
        hybrid_prediction,
        fused_test_score,
    ):

        if bool(prediction) != bool(label):

            errors.append({

                "text":
                    str(text),

                "class":
                    str(cls),

                "true_label":
                    int(label),

                "predicted_label":
                    int(prediction),

                "score":
                    float(score),

                "error_type":
                    (
                        "FN"
                        if label == 1
                        else "FP"
                    ),
            })


    # ----------------------------------------------------------
    # G. Split information
    # ----------------------------------------------------------

    test_families = sorted(
        set(
            row["family"]
            for row, p
            in zip(
                rows,
                part
            )
            if p == "test"
        )
    )

    test_templates = sorted(
        set(
            row["template"]
            for row, p
            in zip(
                rows,
                part
            )
            if p == "test"
        )
    )

    split_info = {

        "seed":
            int(seed),

        "n_train":
            int(train.sum()),

        "n_val":
            int(val.sum()),

        "n_test":
            int(test.sum()),

        "n_positive_test":
            int(
                y[test].sum()
            ),

        "n_negative_test":
            int(
                (y[test] == 0).sum()
            ),

        "target_validation_fpr":
            float(TARGET_FPR),

        "integrity":
            integrity,

        "test_families":
            test_families,

        "test_templates":
            test_templates,

        "photon_best_validation_auc":
            photon_best_val_auc,

        "photon_epochs_run":
            len(photon_history),
    }

    return {

        "seed":
            int(seed),

        "systems":
            systems,

        "drop3":
            drop3,

        "errors":
            errors,

        "split_info":
            split_info,

        "photon_training":
            photon_history,
    }


# ==============================================================
# 15. Summary helpers
# ==============================================================

def mean_std(
    runs,
    system,
    metric,
):

    values = np.array([
        run["systems"][
            system
        ][metric]

        for run in runs
    ])

    return (
        float(values.mean()),
        float(values.std()),
    )


def format_mean_std(
    runs,
    system,
    metric,
):

    mean, std = mean_std(
        runs,
        system,
        metric,
    )

    return (
        f"{mean:.3f}±{std:.3f}"
    )


# ==============================================================
# 16. Main benchmark
# ==============================================================

def main():

    separator = "=" * 100

    print(separator)

    print(
        "SAIS STAGE 2 — DAI SEMANTIC EVALUATOR"
    )

    print(
        "INTENT GENERALISATION EVIDENCE BENCHMARK v3"
    )

    print(separator)

    rows = build_dataset()

    print()

    print(
        f"Dataset actions: {len(rows)}"
    )

    print(
        "Classes:",
        ", ".join(
            TEMPLATES.keys()
        )
    )

    print(
        "Template families:",
        sum(
            len(v)
            for v in TEMPLATES.values()
        )
    )

    print(
        f"Forced unseen test case: {DROP3_TEXT}"
    )

    print(
        f"Target validation FPR: {TARGET_FPR:.2%}"
    )

    print()

    model = load_embedder()

    texts = [
        row["text"]
        for row in rows
    ]

    print()

    print(
        "[1/2] Computing real MiniLM embeddings..."
    )

    X_all = embed_texts(
        model,
        texts
    )

    print(
        f"Embedding shape: {X_all.shape}"
    )

    print()

    print(
        f"[2/2] Running {len(SEEDS)} independent held-out template-family splits..."
    )

    all_runs = []

    for seed in SEEDS:

        print()

        print(
            "-" * 100
        )

        print(
            f"Split seed {seed}"
        )

        print(
            "-" * 100
        )

        result = run_single_split(
            seed,
            rows,
            X_all,
        )

        all_runs.append(
            result
        )

        info = result[
            "split_info"
        ]

        print(
            f"train={info['n_train']} | "
            f"val={info['n_val']} | "
            f"test={info['n_test']}"
        )

        print(
            "Photon epochs:",
            info[
                "photon_epochs_run"
            ]
        )

        print(
            "Photon best validation ROC-AUC:",
            f"{info['photon_best_validation_auc']:.4f}"
        )

        print(
            "Integrity checks: PASS"
        )


    # ==========================================================
    # 17. Aggregate results
    # ==========================================================

    system_names = [
        "baseline@fixed_0.60",
        "baseline@calibrated",
        "photon@fixed_0.60",
        "photon@calibrated",
        "hybrid@calibrated",
    ]

    metrics_to_show = [
        "recall",
        "precision",
        "fpr",
        "specificity",
        "f1",
        "accuracy",
        "roc_auc",
        "pr_auc",
    ]

    print()

    print(separator)

    print(
        "HELD-OUT TEST RESULTS"
    )

    print(
        f"MEAN ± STD ACROSS {len(SEEDS)} TEMPLATE-FAMILY SPLITS"
    )

    print(separator)

    header = (
        f"{'SYSTEM':<26}"
        +
        "".join(
            f"{metric:>15}"
            for metric in metrics_to_show
        )
    )

    print(header)

    print(
        "-" * len(header)
    )

    for system in system_names:

        line = (
            f"{system:<26}"
        )

        for metric in metrics_to_show:

            line += (
                f"{format_mean_std(all_runs, system, metric):>15}"
            )

        print(line)


    # ==========================================================
    # 18. Mean confusion matrix counts
    # ==========================================================

    print()

    print(separator)

    print(
        "MEAN CONFUSION MATRIX COUNTS"
    )

    print(separator)

    for system in system_names:

        tp = np.mean([
            r["systems"][
                system
            ]["TP"]
            for r in all_runs
        ])

        tn = np.mean([
            r["systems"][
                system
            ]["TN"]
            for r in all_runs
        ])

        fp = np.mean([
            r["systems"][
                system
            ]["FP"]
            for r in all_runs
        ])

        fn = np.mean([
            r["systems"][
                system
            ]["FN"]
            for r in all_runs
        ])

        print(
            f"{system:<26}"
            f"TP={tp:6.1f}  "
            f"TN={tn:6.1f}  "
            f"FP={fp:6.1f}  "
            f"FN={fn:6.1f}"
        )


    # ==========================================================
    # 19. Per-class flag rate
    # ==========================================================

    print()

    print(separator)

    print(
        "SHARE OF EACH INTENT CLASS FLAGGED AS DESTRUCTIVE"
    )

    print(
        "(mean across held-out splits)"
    )

    print(separator)

    header = (
        f"{'SYSTEM':<26}"
        +
        "".join(
            f"{cls:>18}"
            for cls in TEMPLATES
        )
    )

    print(header)

    print(
        "-" * len(header)
    )

    for system in system_names:

        line = (
            f"{system:<26}"
        )

        for cls in TEMPLATES:

            key = (
                f"flagged_{cls}"
            )

            value = np.mean([
                r["systems"][
                    system
                ][key]
                for r in all_runs
            ])

            line += (
                f"{value:>17.2%} "
            )

        print(line)


    # ==========================================================
    # 20. DROP_3 generalisation test
    # ==========================================================

    print()

    print(separator)

    print(
        "MANDATORY UNSEEN DROP_3 TEST"
    )

    print(
        repr(DROP3_TEXT)
    )

    print(separator)

    for system in [
        "baseline@calibrated",
        "photon@calibrated",
        "hybrid@calibrated",
    ]:

        halted = sum(
            run["drop3"][
                system
            ]["halted"]

            for run in all_runs
        )

        mean_score = np.mean([
            run["drop3"][
                system
            ]["score"]

            for run in all_runs
        ])

        mean_threshold = np.mean([
            run["drop3"][
                system
            ]["threshold"]

            for run in all_runs
        ])

        print(
            f"{system:<26}"
            f"HALT={halted}/{len(all_runs)} | "
            f"mean score={mean_score:.4f} | "
            f"mean threshold={mean_threshold:.4f}"
        )


    # ==========================================================
    # 21. Hybrid fusion weight
    # ==========================================================

    print()

    print(separator)

    print(
        "HYBRID FUSION"
    )

    print(separator)

    semantic_weights = np.array([
        run["systems"][
            "hybrid@calibrated"
        ]["semantic_weight"]

        for run in all_runs
    ])

    print(
        "Photon semantic weight:"
    )

    print(
        f"  mean={semantic_weights.mean():.3f}"
    )

    print(
        f"  std ={semantic_weights.std():.3f}"
    )

    print(
        "Lexical weight:"
    )

    print(
        f"  mean={(1-semantic_weights).mean():.3f}"
    )


    # ==========================================================
    # 22. Error inspection
    # ==========================================================

    print()

    print(separator)

    print(
        "HYBRID ERRORS — SPLIT 0"
    )

    print(separator)

    errors = (
        all_runs[0][
            "errors"
        ]
    )

    if not errors:

        print(
            "No hybrid test errors on split 0."
        )

    else:

        for error in errors[:30]:

            print(
                f"[{error['error_type']}] "
                f"{error['class']:<14} "
                f"score={error['score']:.4f} | "
                f"{error['text']}"
            )

        if len(errors) > 30:

            print(
                f"... plus {len(errors)-30} additional errors."
            )


    # ==========================================================
    # 23. Integrity evidence
    # ==========================================================

    print()

    print(separator)

    print(
        "DATA LEAKAGE / GENERALISATION CHECKS"
    )

    print(separator)

    all_integrity_passed = True

    for run in all_runs:

        integrity = (
            run[
                "split_info"
            ]["integrity"]
        )

        passed = (

            integrity[
                "family_overlap_train_val"
            ] == 0

            and

            integrity[
                "family_overlap_train_test"
            ] == 0

            and

            integrity[
                "family_overlap_val_test"
            ] == 0

            and

            integrity[
                "exact_text_overlap_train_test"
            ] == 0

            and

            integrity[
                "drop3_test_only"
            ]
        )

        all_integrity_passed &= passed

        print(
            f"seed {run['seed']}: "
            f"{'PASS' if passed else 'FAIL'}"
        )

    print()

    print(
        "OVERALL INTEGRITY:",
        (
            "PASS"
            if all_integrity_passed
            else "FAIL"
        )
    )


    # ==========================================================
    # 24. Save complete audit record
    # ==========================================================

    evidence_package = {

        "benchmark":
            "SAIS Stage 2 Intent Generalisation Evidence Benchmark v3",

        "model":
            MODEL_NAME,

        "positive_class":
            "DESTRUCTIVE",

        "negative_classes": [
            "READ",
            "WRITE_DATA",
            "UPDATE_CONFIG",
        ],

        "configuration": {

            "seeds":
                SEEDS,

            "target_fpr":
                TARGET_FPR,

            "fixed_reference_threshold":
                FIXED_REFERENCE_THRESHOLD,

            "fixed_threshold_definition":
                (
                    "Normalised centroid-distance risk score; "
                    "not raw cosine similarity."
                ),

            "val_families_per_class":
                N_VAL_FAMILIES_PER_CLASS,

            "test_families_per_class":
                N_TEST_FAMILIES_PER_CLASS,

            "drop3_forced_test":
                DROP3_TEXT,

            "adapter": {

                "epochs":
                    ADAPTER_EPOCHS,

                "steps_per_epoch":
                    ADAPTER_STEPS_PER_EPOCH,

                "batch_size":
                    ADAPTER_BATCH_SIZE,

                "margin":
                    ADAPTER_MARGIN,

                "learning_rate":
                    ADAPTER_LR,

                "hidden_dim":
                    ADAPTER_HIDDEN,

                "early_stopping_patience":
                    ADAPTER_PATIENCE,

                "training_objective":
                    (
                        "binary destructive vs non-destructive "
                        "triplet geometry"
                    ),
            },
        },

        "runs":
            all_runs,
    }

    with open(
        RESULT_JSON,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            evidence_package,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print()

    print(separator)

    print(
        f"Complete evidence record saved to: {RESULT_JSON}"
    )

    print(separator)


# ==============================================================
# 25. Execute
# ==============================================================

main()

SAIS STAGE 2 — DAI SEMANTIC EVALUATOR
INTENT GENERALISATION EVIDENCE BENCHMARK v3

Dataset actions: 490
Classes: READ, WRITE_DATA, UPDATE_CONFIG, DESTRUCTIVE
Template families: 49
Forced unseen test case: DELETE FROM accounts WHERE 1=1
Target validation FPR: 2.00%

Loading model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


[1/2] Computing real MiniLM embeddings...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Embedding shape: (490, 384)

[2/2] Running 5 independent held-out template-family splits...

----------------------------------------------------------------------------------------------------
Split seed 0
----------------------------------------------------------------------------------------------------
train=250 | val=120 | test=120
Photon epochs: 13
Photon best validation ROC-AUC: 1.0000
Integrity checks: PASS

----------------------------------------------------------------------------------------------------
Split seed 1
----------------------------------------------------------------------------------------------------
train=250 | val=120 | test=120
Photon epochs: 13
Photon best validation ROC-AUC: 1.0000
Integrity checks: PASS

----------------------------------------------------------------------------------------------------
Split seed 2
----------------------------------------------------------------------------------------------------
train=250 | val=120 | test=120
Photon 